In [1]:
import os
import xarray as xr
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pystac_client
from scipy import ndimage as ndi
from distributed import LocalCluster
from pyproj import Transformer
import json
import geopandas as gpd
from shapely.geometry import Point, box
from shapely.ops import unary_union
from dask_gateway import Gateway
import pathlib
import re
from dask.distributed import Client

def extract_time(ds):
    date_format = "%Y%m%dT%H%M%S"
    filename = ds.encoding["source"]
    date_str = os.path.basename(filename).split("_")[2]
    time = datetime.strptime(date_str, date_format)
    return ds.assign_coords(time=time)

def circular_kernel(radius: int) -> np.ndarray:
    """Your existing circular kernel function"""
    r = int(radius)
    y, x = np.ogrid[-r:r+1, -r:r+1]
    k = (x*x + y*y) <= (r*r)
    return k.astype(np.float32)

def _dilate_with_convolve(mask_2d: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """Your existing dilation function"""
    if mask_2d.dtype != np.float32:
        mask_2d = mask_2d.astype(np.float32, copy=False)
    conv = ndi.convolve(mask_2d, kernel, mode="constant", cval=0.0)
    return conv > 0.0

def _apply_dilation_block(scl_block: np.ndarray, k1: np.ndarray, k2: np.ndarray) -> np.ndarray:
    """Your existing dilation application function"""
    scl = scl_block.astype(np.int16, copy=False)
    mask1 = (scl != 2) & (scl != 4) & (scl != 5) & (scl != 6) & (scl != 7)
    mask2 = (scl == 3) | (scl == 8) | (scl == 9) | (scl == 10) | (scl == 11)
    dil1 = _dilate_with_convolve(mask1, k1)
    dil2 = _dilate_with_convolve(mask2, k2)
    return dil1 | dil2

def apply_scl_dilation_to_dataset(ds: xr.Dataset, kernel1: np.ndarray, kernel2: np.ndarray) -> xr.Dataset:
    """
    Apply SCL dilation masking to the entire dataset using your existing logic.
    This is the vectorized version of your mask_scl_dilation function.
    """
    if "scl" not in ds:
        raise ValueError(f"'scl' not found in dataset variables: {list(ds.data_vars)}")
    
    scl = ds["scl"]
    
    # Apply dilation using xarray's apply_ufunc
    dilated = xr.apply_ufunc(
        _apply_dilation_block,
        scl,
        input_core_dims=[["y", "x"]],
        output_core_dims=[["y", "x"]],
        kwargs={"k1": kernel1.astype(np.float32), "k2": kernel2.astype(np.float32)},
        dask="parallelized",
        vectorize=True,
        output_dtypes=[bool],
        dask_gufunc_kwargs={"allow_rechunk": True},
    ).rename("dilated_mask")
    
    # Apply mask to all bands except SCL
    bands_to_mask = [v for v in ds.data_vars if v != "scl"]
    
    out = ds.copy()
    for var in bands_to_mask:
        da = out[var]
        if np.issubdtype(da.dtype, np.integer):
            da = da.astype(np.float32)
        out[var] = da.where(~dilated)
    
    # Remove time steps where all data is masked
    keep = xr.concat([out[v].notnull().any(["y", "x"]) for v in bands_to_mask], dim="vars").any("vars")
    out = out.sel(time=keep)
    
    return out.drop_vars("scl")

def extract_timeseries_from_datacube(datacube, scl, b11, points_per_type, kernel1, kernel2, buffer_m=20):
    """
    Extract time series from prebuilt datacube with proper SCL dilation masking.
    """
    # Convert points to GeoDataFrame
    gdf = _points_gdf_from_buffers(points_per_type)
    
    # Ensure all datasets have the same CRS and are properly aligned
    datacube = datacube.rio.write_crs("EPSG:32631")
    scl = scl.rio.write_crs("EPSG:32631")
    b11 = b11.rio.write_crs("EPSG:32631")
    
    # Resample b11 and scl to match 10m resolution
    print("Resampling 20m bands to 10m resolution...")
    b11_10m = b11.interp_like(datacube, method="nearest")
    scl_10m = scl.interp_like(datacube, method="nearest")
    
    # Combine all bands into one dataset
    combined_ds = xr.Dataset({
        "b04": datacube["b04"],
        "b08": datacube["b08"],
        "b11": b11_10m["b11"],
        "scl": scl_10m["scl"]
    })
    
    # Apply SCL dilation masking to the entire datacube
    print("Applying SCL dilation masking...")
    masked_ds = apply_scl_dilation_to_dataset(combined_ds, kernel1, kernel2)
    
    # Compute NDVI for the entire datacube
    print("Computing NDVI...")
    ndvi = compute_ndvi_dataset(masked_ds)
    masked_ds = masked_ds.assign(ndvi=ndvi)
    
    # Extract time series for each point
    results = {}
    print(f"Extracting time series for {len(gdf)} points...")
    
    for idx, row in gdf.iterrows():
        fid = int(row["feature_index"])
        crop = row["name"]
        geometry = row["geometry"]
        
        print(f"Processing {crop} point {fid}...")
        
        # Extract time series for this point using buffer method (more robust)
        point_ts = extract_buffer_timeseries(masked_ds, geometry, buffer_m)
        
        if point_ts is not None and len(point_ts) > 0:
            results[(crop, fid)] = point_ts
            print(f"  Extracted {len(point_ts)} time points")
        else:
            print(f"  No valid data for point {fid}")
    
    return results

def compute_ndvi_dataset(ds):
    """Compute NDVI for entire dataset."""
    nir = ds.b08.astype("float32")
    red = ds.b04.astype("float32")
    ndvi = (nir - red) / (nir + red)
    ndvi = ndvi.where((nir + red) != 0, np.nan)
    return ndvi


kernel1 = circular_kernel(radius=8)
kernel2 = circular_kernel(radius=100)

# Simplest way - creates everything automatically
client = Client() 

# ---------- paths ----------
OUT_BASE = pathlib.Path("outputs/samples_zarr")
OUT_BASE.mkdir(parents=True, exist_ok=True)
# Define AOI and convert to raster CRS
spatial_extent = {
    "west": 3.2865,
    "south": 50.7589,
    "east": 3.7752,
    "north": 50.9842,
}
'''
spatial_extent = {
    "west": 3.4292229675292973,
    "south": 50.776963820264364,
    "east": 3.5005130004882817,
    "north": 50.82166185876709,
}
'''

bbox_4326 = [
    spatial_extent["west"],
    spatial_extent["south"],
    spatial_extent["east"],
    spatial_extent["north"],
]

# Convert AOI to UTM 33N (same as Sentinel-2 data)
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32631", always_xy=True)
west_utm, south_utm = transformer.transform(
    spatial_extent["west"], spatial_extent["south"]
)
east_utm, north_utm = transformer.transform(
    spatial_extent["east"], spatial_extent["north"]
)

# Spatial slice parameters
x_slice = slice(west_utm, east_utm)
y_slice = slice(north_utm, south_utm)

# Connect to the STAC catalog
catalog = pystac_client.Client.open("https://stac.core.eopf.eodc.eu")

# Search for Sentinel-2 L2A items within a specific bounding box and date range
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox_4326,
    datetime="2018-11-01/2020-02-01",
)

# Retrieve the list of matching items
items = list(search.items())
hrefs = [item.assets["product"].href for item in items]

datacube = xr.open_mfdataset(
    hrefs,
    engine="zarr",
    chunks={},
    group="/measurements/reflectance/r10m",
    concat_dim="time",
    combine="nested",
    preprocess=extract_time,
    mask_and_scale=True,
).sortby("time", ascending=True).sel(x=x_slice, y=y_slice)

scl = xr.open_mfdataset(
    hrefs,
    engine="zarr",
    chunks={},
    group="/conditions/mask/l2a_classification/r20m",  # Adjust if necessary
    concat_dim="time",
    combine="nested",
    preprocess=extract_time,
    mask_and_scale=True,
).sortby("time", ascending=True)[["scl"]]


b11 = xr.open_mfdataset(
    hrefs,
    engine="zarr",
    chunks={},
    group="/measurements/reflectance/r20m",  # Adjust if necessary
    concat_dim="time",
    combine="nested",
    preprocess=extract_time,
    mask_and_scale=True,
).sortby("time", ascending=True)[["b11"]]


crops = {"maize":1200,"potatos":5100,"sugarbeet":8100,"barley":1500,"soy":4100}
crop_samples = {name:gpd.read_file("resources/"+ name + "_2019.geojson") for name,code in crops.items()}
kernel1 = circular_kernel(radius=8)
kernel2 = circular_kernel(radius=100)

datacube = datacube.rio.write_crs("EPSG:32631")  # ensure CRS

datacube


<xarray.Dataset> Size: 50GB
Dimensions:      (time: 179, y: 2530, x: 3420)
Coordinates:
  * x            (x) float32 14kB 5.202e+05 5.202e+05 ... 5.544e+05 5.544e+05
  * y            (y) float32 10kB 5.648e+06 5.648e+06 ... 5.623e+06 5.623e+06
  * time         (time) datetime64[ns] 1kB 2018-11-02T10:52:09 ... 2020-01-31...
    spatial_ref  int64 8B 0
Data variables:
    b02          (time, y, x) float64 12GB dask.array<chunksize=(1, 325, 1637), meta=np.ndarray>
    b03          (time, y, x) float64 12GB dask.array<chunksize=(1, 325, 1637), meta=np.ndarray>
    b04          (time, y, x) float64 12GB dask.array<chunksize=(1, 325, 1637), meta=np.ndarray>
    b08          (time, y, x) float64 12GB dask.array<chunksize=(1, 325, 1637), meta=np.ndarray>

In [2]:
# --- helpers ---
def _raster_bounds_polygon_32631(ds):
    """Shapely box of the datacube extent (expects x/y in EPSG:32631)."""
    xmin = float(ds.x.min())
    xmax = float(ds.x.max())
    ymin = float(ds.y.min())
    ymax = float(ds.y.max())
    return box(min(xmin, xmax), min(ymin, ymax), max(xmin, xmax), max(ymin, ymax))

def _random_point_in_geom(geom, rng=None, max_tries=10_000):
    """Rejection sample one point inside (multi)polygon."""
    if rng is None:
        rng = np.random.default_rng()
    minx, miny, maxx, maxy = geom.bounds
    for _ in range(max_tries):
        x = rng.uniform(minx, maxx)
        y = rng.uniform(miny, maxy)
        p = Point(x, y)
        if geom.contains(p):
            return p
    raise RuntimeError("Failed to sample a point inside geometry within max_tries.")

def simple_random_sampling_within_bounds(crop_samples, datacube, points_per_crop=5, buffer_m=20):
    """
    Simple random sampling: find polygons within datacube bounds and pick 5 random points from them.
    Returns {crop_name: GeoJSON FeatureCollection (EPSG:32631)}
    """
    # Get datacube bounds
    raster_poly = _raster_bounds_polygon_32631(datacube)
    
    points_per_type = {}
    
    for crop_name, gdf in crop_samples.items():
        # Ensure we have a valid GeoDataFrame with geometry
        if gdf.crs is None:
            gdf = gdf.set_crs("EPSG:4326")
        if gdf.crs.to_epsg() != 32631:
            gdf = gdf.to_crs("EPSG:32631")
        
        # Find polygons that intersect with datacube bounds
        polygons_within_bounds = []
        for idx, geometry in enumerate(gdf.geometry):
            if geometry.intersects(raster_poly):
                polygons_within_bounds.append((idx, geometry))
        
        if not polygons_within_bounds:
            print(f"Warning: No polygons for {crop_name} intersect with datacube bounds")
            continue
        
        print(f"{crop_name}: {len(polygons_within_bounds)} polygons within datacube bounds")
        
        # If we have polygons within bounds, pick one that fits well
        selected_polygons = []
        for idx, geometry in polygons_within_bounds:
            # Check if this polygon is mostly within the datacube bounds
            intersection = geometry.intersection(raster_poly)
            if intersection.area > 0.5 * geometry.area:  # At least 50% of polygon is within bounds
                selected_polygons.append((idx, geometry))
                # If we found one good polygon, use it for all 5 points
                if len(selected_polygons) >= 1:  # Just need one good polygon
                    break
        
        if not selected_polygons:
            # If no polygon is mostly within bounds, use the largest intersection
            largest_area = 0
            best_polygon = None
            for idx, geometry in polygons_within_bounds:
                intersection = geometry.intersection(raster_poly)
                if intersection.area > largest_area:
                    largest_area = intersection.area
                    best_polygon = (idx, geometry)
            if best_polygon:
                selected_polygons = [best_polygon]
        
        if not selected_polygons:
            print(f"Warning: No suitable polygon found for {crop_name}")
            continue
        
        # Use the first selected polygon to sample all 5 points
        poly_idx, selected_polygon = selected_polygons[0]
        print(f"Using polygon {poly_idx} for {crop_name} (area: {selected_polygon.area:.0f} m²)")
        
        # Sample 5 random points from this single polygon
        buffered_points = []
        for i in range(points_per_crop):
            random_point = _random_point_in_geom(selected_polygon)
            buffered_point = random_point.buffer(buffer_m)
            buffered_points.append(buffered_point)
        
        # Create GeoDataFrame with buffered points
        result_gdf = gpd.GeoDataFrame({
            'name': [crop_name] * len(buffered_points),
            'geometry': buffered_points,
            'polygon_index': [poly_idx] * len(buffered_points),  # Track which polygon was used
            'point_index': list(range(len(buffered_points)))  # Track point number
        }, crs="EPSG:32631")
        
        # Convert to GeoJSON string
        points_per_type[crop_name] = result_gdf.to_json()
    
    return points_per_type

# Use the simple sampling
points_per_type = simple_random_sampling_within_bounds(
    crop_samples=crop_samples,
    datacube=datacube,
    points_per_crop=1,
    buffer_m=20
)

print(points_per_type.keys())

maize: 1 polygons within datacube bounds
Using polygon 1 for maize (area: 57299 m²)
potatos: 1 polygons within datacube bounds
Using polygon 2 for potatos (area: 30128 m²)
sugarbeet: 4 polygons within datacube bounds
Using polygon 0 for sugarbeet (area: 17616 m²)
barley: 1 polygons within datacube bounds
Using polygon 8 for barley (area: 13771 m²)
soy: 2 polygons within datacube bounds
Using polygon 3 for soy (area: 10025 m²)
dict_keys(['maize', 'potatos', 'sugarbeet', 'barley', 'soy'])


In [3]:
points_per_type

{'maize': '{"type": "FeatureCollection", "features": [{"id": "0", "type": "Feature", "properties": {"name": "maize", "polygon_index": 1, "point_index": 0}, "geometry": {"type": "Polygon", "coordinates": [[[520495.86723829986, 5623194.086479645], [520495.7709328333, 5623192.126136838], [520495.4829439079, 5623190.184673205], [520495.0060450145, 5623188.2807861], [520494.34482895007, 5623186.432810998], [520493.50566358684, 5623184.658544908], [520492.49663054594, 5623182.975074985], [520491.32744736713, 5623181.398613962], [520490.0093739236, 5623179.944344021], [520488.55510398315, 5623178.626270578], [520486.97864296025, 5623177.457087399], [520485.2951730364, 5623176.448054358], [520483.52090694715, 5623175.608888995], [520481.67293184495, 5623174.947672931], [520479.7690447402, 5623174.470774037], [520477.82758110645, 5623174.1827851115], [520475.86723829986, 5623174.086479645], [520473.9068954933, 5623174.1827851115], [520471.96543185954, 5623174.470774037], [520470.0615447548, 562

In [4]:
import warnings


def _sanitize_netcdf_attrs(ds: xr.Dataset) -> xr.Dataset:
    """
    Remove or stringify attrs that netCDF can't store (e.g., dicts, custom objects).
    Also drops private attrs starting with '_' (like '_eopf_attrs') to be safe.
    """
    def clean(mapping):
        bad_keys = []
        for k, v in list(mapping.items()):
            if k.startswith("_"):
                bad_keys.append(k)
                continue
            if isinstance(v, (str, bytes, int, float, bool, np.number, np.ndarray, list, tuple)):
                # ok
                continue
            # dicts or anything else: either drop or stringify; here we drop
            bad_keys.append(k)
        for k in bad_keys:
            mapping.pop(k, None)

    ds = ds.copy(deep=False)
    clean(ds.attrs)
    for name, da in ds.variables.items():
        clean(da.attrs)
    return ds

    
def _align_to_datacube_grid(dc: xr.Dataset, x: float, y: float):
    """Return the nearest datacube x/y coordinates and their integer indices."""
    # nearest index lookup (works lazily with Dask)
    ix = int(abs(dc.x - x).argmin())
    iy = int(abs(dc.y - y).argmin())
    xg = float(dc.x.isel(x=ix))
    yg = float(dc.y.isel(y=iy))
    return ix, iy, xg, yg

def _subset_window(dc: xr.Dataset, x: float, y: float, buffer_m: float | int):
    """Subset a small x/y window around (x,y) from datacube using a metric buffer."""
    if buffer_m is None or buffer_m <= 0:
        # one pixel only
        ix, iy, xg, yg = _align_to_datacube_grid(dc, x, y)
        return dc.isel(x=slice(ix, ix+1), y=slice(iy, iy+1))
    # convert meters to number of 10 m pixels (ceil and add margin)
    px = int(np.ceil(buffer_m / 10.0))
    ix, iy, xg, yg = _align_to_datacube_grid(dc, x, y)
    xs = slice(max(ix - px, 0), ix + px + 1)
    ys = slice(max(iy - px, 0), iy + px + 1)
    return dc.isel(x=xs, y=ys)

def _interp_like_local(src: xr.Dataset | xr.DataArray, like: xr.Dataset) -> xr.Dataset | xr.DataArray:
    """Cheap local resampling: resample only to the tiny 'like' grid."""
    # Use nearest to avoid introducing NaNs on tiny windows; it’s fast and robust
    return src.interp_like(like, method="nearest")

def _combine_and_reduce(dc_win: xr.Dataset, scl_win: xr.Dataset, b11_win: xr.Dataset, reduce: str = "median"):
    """
    Combine bands and reduce spatial dims to a single time series.
    reduce: 'median' | 'mean' | 'nearest'
    """
    ds = xr.Dataset({
        "b04": dc_win["b04"],
        "b08": dc_win["b08"],
        "b11": b11_win["b11"],
        "scl": scl_win["scl"]
    })
    # NDVI
    ndvi = (ds.b08.astype("float32") - ds.b04.astype("float32")) / (ds.b08 + ds.b04)
    ndvi = ndvi.where((ds.b08 + ds.b04) != 0)
    ds = ds.assign(ndvi=ndvi)

    # collapse x,y to single point series
    if reduce == "nearest":
        ds = ds.isel(x=0, y=0)
    elif reduce == "mean":
        ds = ds.mean(dim=("y", "x"), skipna=True)
    else:
        ds = ds.median(dim=("y", "x"), skipna=True)

    return ds

def _geojsons_to_point_df(points_per_type: dict[str, str]) -> gpd.GeoDataFrame:
    """Turn your {crop: geojson_str} into a single EPSG:32631 GeoDataFrame with columns: name, point_index, geometry."""
    rows = []
    for crop, gj in points_per_type.items():
        gdf = gpd.read_file(gj)
        if gdf.crs is None:
            gdf = gdf.set_crs("EPSG:32631")
        elif gdf.crs.to_epsg() != 32631:
            gdf = gdf.to_crs("EPSG:32631")
        # Your sampling produced buffered polygons; use their centroids as the 'point'
        gdf = gdf.assign(name=crop, point_index=gdf.get("point_index", pd.Series(range(len(gdf)))))
        gdf["geometry"] = gdf.geometry.centroid
        rows.append(gdf[["name", "point_index", "geometry"]])
    if not rows:
        return gpd.GeoDataFrame(columns=["name", "point_index", "geometry"], crs="EPSG:32631")
    return pd.concat(rows, ignore_index=True)

def extract_point_series_for_crops(
    datacube: xr.Dataset,
    scl: xr.Dataset,
    b11: xr.Dataset,
    points_per_type: dict[str, str],
    fallback_buffer_m: int = 20,
    reduce: str = "median",
    out_dir: pathlib.Path | str = "outputs/samples_netcdf",
) -> dict[tuple[str, int], str]:
    """
    For each crop point:
      1) try point-only resample (1 pixel)
      2) if any band is entirely NaN through time, retry with fallback_buffer_m window
      3) reduce to 1-D time series and save <crop>.nc
    Returns {(crop, point_index): path}
    """
    out_dir = pathlib.Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    pts = _geojsons_to_point_df(points_per_type)
    if pts.empty:
        warnings.warn("No points to process.")
        return {}

    results = {}

    # ensure CRS is written (already done in your script)
    datacube = datacube.rio.write_crs("EPSG:32631")
    scl = scl.rio.write_crs("EPSG:32631")
    b11 = b11.rio.write_crs("EPSG:32631")

    for _, row in pts.iterrows():
        crop = row["name"]
        pidx = int(row["point_index"])
        x = float(row.geometry.x)
        y = float(row.geometry.y)

        # 1) single-pixel attempt (cheapest)
        dc_win = _subset_window(datacube, x, y, buffer_m=0)
        scl_loc = _interp_like_local(scl, dc_win)
        b11_loc = _interp_like_local(b11, dc_win)
        ds_point = _combine_and_reduce(dc_win, scl_loc, b11_loc, reduce="nearest")

        # Check NaNs (if any key variable is fully NaN through time)
        needs_fallback = False
        for var in ["b04", "b08", "b11", "ndvi"]:
            da = ds_point[var]
            # all-NaN across all timesteps?
            if da.isnull().all():
                needs_fallback = True
                break

        # 2) fallback with tiny spatial window (e.g., 20 m)
        if needs_fallback and fallback_buffer_m and fallback_buffer_m > 0:
            dc_win = _subset_window(datacube, x, y, buffer_m=fallback_buffer_m)
            scl_loc = _interp_like_local(scl, dc_win)
            b11_loc = _interp_like_local(b11, dc_win)
            ds_point = _combine_and_reduce(dc_win, scl_loc, b11_loc, reduce=reduce)

        # Final sanity: drop all-NaN timesteps
        keep = xr.concat([ds_point[v].notnull() for v in ["b04", "b08", "b11", "ndvi"]], "v").any("v")
        ds_point = ds_point.sel(time=keep)

        # Save one NetCDF per crop (append if multiple points/crops later)
        out_path = out_dir / f"{crop}.nc"
        mode = "a" if out_path.exists() else "w"

        # To keep files tidy when appending, add a coord 'sample' to differentiate points if needed
        ds_to_write = ds_point.expand_dims(sample=[pidx])

        # (1) shrink the dask graph for this tiny window
        ds_to_write = ds_to_write.persist()  # tiny window => cheap, reduces graph bloat
        
        # (2) sanitize attrs so netCDF is happy
        ds_to_write = _sanitize_netcdf_attrs(ds_to_write)

        # Keep lazy where possible; .to_netcdf handles dask
        ds_to_write.to_netcdf(out_path, mode="w" if mode == "w" else "a")

        results[(crop, pidx)] = str(out_path)
        print(f"[{crop}:{pidx}] -> {out_path}")

    return results


In [ ]:
# Extract + save time-series per crop (1 sample per crop as configured)
saved = extract_point_series_for_crops(
    datacube=datacube,
    scl=scl,
    b11=b11,
    points_per_type=points_per_type,
    fallback_buffer_m=20,   # try single pixel first, then 20 m window if needed
    reduce="median",        # spatial reduce when using a window ('median'|'mean'|'nearest')
    out_dir="outputs/samples_netcdf"
)

print("Saved files:", saved)


/home/sdhinakaran/micromamba/envs/eopf-zarr/lib/python3.11/site-packages/distributed/client.py:3371: UserWarning: Sending large graph of size 24.88 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/home/sdhinakaran/micromamba/envs/eopf-zarr/lib/python3.11/site-packages/distributed/client.py:3371: UserWarning: Sending large graph of size 24.88 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
